# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime -> Change runtime type -> T4 GPU.
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TrKhuyn/K4-Track-4-Day10-VisionLanguageModel.git"
# Guide default 2/4. For a tight T4 session, select 1/8 BEFORE Run all;
# the same effective batch (8) and profile apply to E1, E2, E3 and E4.
TRAIN_BATCH_SIZE, GRAD_ACCUM = 2, 4

assert Path("/content").exists(), "Run this notebook on Google Colab."
repo = Path("/content/day10-vlm-repo")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
ROOT = Path.cwd().resolve()
assert (ROOT/"GUIDE.md").is_file()
assert all((ROOT/name).is_file() for name in ("run_lab.py", "lab_runtime.py", "lab_reward.py"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
for name in ("run_lab", "lab_runtime", "lab_reward"):
    sys.modules.pop(name, None)
globals().pop("_DAY10_LAB", None)
globals().pop("_DAY10_REQUEST", None)
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r",
                "requirements-colab.txt", "pytest"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchao"], check=True)

subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "student_checks"], check=True)

In [ ]:
# Secrets are read into the environment; keys are never printed or saved.
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/day10-vlm")

## 1. Dữ liệu: tải và chia split

In [ ]:
import json
split_files = [ROOT/f"data/splits/{name}.jsonl" for name in ("train", "val", "test")]
reuse_data = (ROOT/"data/splits/meta.json").is_file() and all(p.is_file() for p in split_files)
if reuse_data:
    for path in split_files:
        rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if not rows or not all((ROOT/"data"/row["image_path"]).is_file() for row in rows):
            reuse_data = False
            break
if reuse_data:
    print("Using the existing prepared dataset; Lab will validate counts and sessions.")
else:
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--kaggle"], check=True)

In [ ]:
_DAY10_STAGE_OK = False
%run -i run_lab.py --stage data
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
_DAY10_STAGE_OK = False
# Validate normalized labels, deterministic y1/x1 ordering, assistant-only loss,
# and a real E3 optimizer update before the long E0-E5 runs.
%run -i run_lab.py --stage labels
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

## 3. E0: zero-shot

In [ ]:
_DAY10_STAGE_OK = False
%run -i run_lab.py --stage e0
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

In [ ]:
_DAY10_STAGE_OK = False
%run -i run_lab.py --stage raw-e0
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

In [ ]:
_DAY10_STAGE_OK = False
%run -i run_lab.py --stage score-e0
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
_DAY10_STAGE_OK = False
%run -i run_lab.py --stage mask
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
_DAY10_STAGE_OK = False
%run -i run_lab.py --stage setup
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

## 6. Vòng huấn luyện

In [ ]:
_DAY10_STAGE_OK = False
# Train E1/E2/E3 (300 optimizer updates each), reload their saved adapters /
# projectors, score full validation and test splits, then select by validation.
%run -i run_lab.py --stage sft
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

## 7. Suy luận và chấm E1–E3

In [ ]:
_DAY10_STAGE_OK = False
%run -i run_lab.py --stage summary
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
_DAY10_STAGE_OK = False
# E4 changes data_size only: 142 train images vs the original 1422-image run.
%run -i run_lab.py --stage e4
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

In [ ]:
_DAY10_STAGE_OK = False
# E5: TRL 1.9.2 GRPO, four completions/image, 60 updates, lr=5e-6,
# beta=0, no vLLM, initialized from the best SFT adapter + projector.
%run -i run_lab.py --stage e5
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."

In [ ]:
_DAY10_STAGE_OK = False
# Save seven official evidence files, six prediction figures, raw outputs and
# measured logs. Download the ZIP; adapters/checkpoints remain on Drive.
%run -i run_lab.py --stage export
assert _DAY10_STAGE_OK, "Stage failed; review the traceback above before continuing."